# Curso: Redes Neurais para Processamento de Linguagem Natural

Prof. [Denilson Alves Pereira](https://sites.google.com/ufla.br/denilsonpereira) <br>
Departamento de Ciência da Computação (DCC) <br>
Instituto de Ciências Exatas e Tecnológicas (ICET) <br> 
Universidade Federal de Lavras (UFLA)

**Aluno:** Marcos Carvalho Ferreira - **Matrícula:** 2026160060

# Atividade Prática 02

**Instruções:**
1. Siga os passos indicados em cada célula abaixo para completar a atividade.
2. Você deve inserir código somente entre as linhas marcadas com **INICIE O CÓDIGO AQUI** e **TERMINE O CÓDIGO AQUI**. Há uma indicação de quantas linhas de código são necessárias.
3. Em alguns pontos, confira o resultado esperado conforme marcado com **SAÍDA ESPERADA**.

**Tempo estimado para execução**: 2,0 horas

Versão: Maio, 2024

## O Problema a ser Resolvido

O objetivo da atividade é elaborar um modelo de PLN para a tarefa de classificação de textos usando o Modelo BERT. <br>

Classificação de texto é uma tarefa de PLN que atribui um rótulo (ou classe) a um texto. <br>
Nesta atividade, você vai usar o dataset HateBR para classificar se um texto possui ou não uma linguagem ofensiva (classificação binária) e o nível de ofensividade (levemente ofensivo, moderado, alto). <br>

Você vai praticar as seguintes habilidades:
- Usar a biblioteca Datasets
- Pré-processar textos usando um tokenizador
- Fazer um ajuste fino (finetune) do modelo BERTimbau usando o dataset HateBR
- Avaliar o modelo ajustado e usá-lo para fazer inferências

Referências: <br>
https://huggingface.co/docs/transformers/tasks/sequence_classification <br>
https://huggingface.co/course/chapter1/1

## Pacotes

In [1]:
import numpy as np # scientific computing
import tensorflow as tf  #  numerical computation using data flow graphs
from transformers import AutoTokenizer, DataCollatorWithPadding, pipeline  # API Transformer do Hugging Face
from transformers import create_optimizer, TFAutoModelForSequenceClassification  # API Transformer do Hugging Face
from transformers.keras_callbacks import KerasMetricCallback  # API Transformer do Hugging Face
from datasets import load_dataset  # função para carga de datasets da biblioteca Datasets
import evaluate  # biblioteca de métricas de avaliação experimental
import matplotlib.pyplot as plt # scientific plotting library
from datetime import datetime # data e hora
from sklearn.metrics import accuracy_score, precision_recall_fscore_support # metricas de avaliação
from collections import Counter

2026-09-21 11:14:20.220741: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-09-21 11:14:20.222610: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-09-21 11:14:20.245584: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-09-21 11:14:20.245609: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-09-21 11:14:20.245626: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to regi

## Pré-Processamento dos Dados

### Carga do Dataset

Dataset (base de dados): HateBR da biblioteca Datasets (https://huggingface.co/datasets/ruanchaves/hatebr)

O corpus HateBR contém comentários sobre política, coletados do Instagram. Ele contém dados rotulados sobre linguagem ofensiva e discurso de ódio, com textos em português do Brasil.

O dataset é composto por 7.000 documentos anotados de acordo com três diferentes camadas: linguagem ofensiva (comentário ofensivo versus não ofensivo), nível de ofensa (levemente ofensivo, moderado, alto) e nove grupos de discursos de ódio (xenofobia, racismo, homofobia, sexismo, intolerância religiosa, partidário, apologia à ditadura, anti-semitismo e gordofobia)

A biblioteca 🤗 Datasets fornece um comando para baixar e colocar em cache um dataset do Hub do Hugging Face.<br>
O objeto DatasetDict resultante da carga contém as partições do conjunto de dados, e cada partição contém um conjunto de atributos (features) e uma variável com o número de linhas que ela possui.

Hub de datasets do Hugging Face: https://huggingface.co/datasets

In [2]:
# Carrega o dataset HateBR da biblioteca Datasets
db = load_dataset("ruanchaves/hatebr")
# Exibe a estrutura do dataset
db

DatasetDict({
    train: Dataset({
        features: ['instagram_comments', 'offensive_language', 'offensiveness_levels', 'antisemitism', 'apology_for_the_dictatorship', 'fatphobia', 'homophobia', 'partyism', 'racism', 'religious_intolerance', 'sexism', 'xenophobia', 'offensive_&_non-hate_speech', 'non-offensive', 'specialist_1_hate_speech', 'specialist_2_hate_speech', 'specialist_3_hate_speech'],
        num_rows: 4480
    })
    validation: Dataset({
        features: ['instagram_comments', 'offensive_language', 'offensiveness_levels', 'antisemitism', 'apology_for_the_dictatorship', 'fatphobia', 'homophobia', 'partyism', 'racism', 'religious_intolerance', 'sexism', 'xenophobia', 'offensive_&_non-hate_speech', 'non-offensive', 'specialist_1_hate_speech', 'specialist_2_hate_speech', 'specialist_3_hate_speech'],
        num_rows: 1120
    })
    test: Dataset({
        features: ['instagram_comments', 'offensive_language', 'offensiveness_levels', 'antisemitism', 'apology_for_the_dictato

In [3]:
# Mostra os rótulos do dataset de treino para a classe 'linguagem ofensiva'
db["train"].features["offensive_language"]

Value(dtype='bool', id=None)

In [4]:
# Mostra as três primeiras revisões do dataset de teste
db["test"][0:5]

{'instagram_comments': ['Mais um lixo',
  'Essa mulher é doente.pilantra!',
  'Vagabunda. Comunista. Mentirosa. O povo chileno nao merece uma desgraça desta.',
  'Besta quadrada.',
  'Hipocritas'],
 'offensive_language': [True, True, True, True, True],
 'offensiveness_levels': [1, 3, 3, 2, 2],
 'antisemitism': [False, False, False, False, False],
 'apology_for_the_dictatorship': [False, False, False, False, False],
 'fatphobia': [False, False, False, False, False],
 'homophobia': [False, False, False, False, False],
 'partyism': [False, False, True, False, False],
 'racism': [False, False, False, False, False],
 'religious_intolerance': [False, False, False, False, False],
 'sexism': [False, False, True, False, False],
 'xenophobia': [False, False, False, False, False],
 'offensive_&_non-hate_speech': [True, True, False, True, True],
 'non-offensive': [False, False, False, False, False],
 'specialist_1_hate_speech': [False, False, False, False, False],
 'specialist_2_hate_speech': [Fal

In [5]:
# Renomeia o atributo alvo (classe) para 'label', que é o nome padrão usado pelo modelo de classificação
db = db.rename_column('offensive_language', 'label')
# Remove os atributos que não serão usados nesta aplicação
db = db.remove_columns(['offensiveness_levels', 'antisemitism', 'apology_for_the_dictatorship', 'fatphobia', 'homophobia', 'partyism', 'racism', 'religious_intolerance', 'sexism', 'xenophobia', 'offensive_&_non-hate_speech', 'non-offensive', 'specialist_1_hate_speech', 'specialist_2_hate_speech', 'specialist_3_hate_speech'])

### Tokenização pelo Modelo BERTimbau

O modelo BERTimbau (https://huggingface.co/neuralmind/bert-base-portuguese-cased) é um modelo BERT pré-treinado para a língua portuguesa do Brasil.

É necessário carregar o tokenizador BERTimbau para pré-processar o atributo com os comentários do dataset.

In [6]:
# Carrega o tokenizador BERTimbau
checkpoint = "neuralmind/bert-base-portuguese-cased" # BERTimbau base, em português, com tokens não convertidos para minúsculas
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

Cria uma função de pré-processamento para tokenizar o texto e truncar as sequências maiores do que o comprimento máximo de entrada do modelo BERTimbau:

In [7]:
### INICIE O CÓDIGO AQUI ### (2 linhas de código)
def preprocess_function(examples):
    return tokenizer(examples["instagram_comments"], truncation=True)
### TERMINE O CÓDIGO AQUI ###

A função "map" da biblioteca 🤗 Datasets pode ser usada para aplicar a função "preprocess_function", definida acima, para todo o dataset. O argumento "batched=True" indica que o processamento deve ser feito em lote de múltiplos elementos do dataset, o que torna a função "map" mais rápida.

In [8]:
tokenized_db = db.map(preprocess_function, batched=True)

Map: 100%|██████████| 1400/1400 [00:00<00:00, 76372.15 examples/s]


"DataCollatorWithPadding" cria um lote usando uma lista de elementos de um dataset como entrada. Durante o processamento, podem ser aplicados "padding", por exemplo. Na chamada abaixo, o argumento "padding" é por padrão igual a "True", o que faz o preenchimento (padding) pela sequência mais longa no lote. É mais eficiente preencher dinamicamente as sentenças pelo comprimento mais longo no lote durante o agrupamento (collation) do que preencher todo o dataset com o comprimento máximo.

In [9]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

## Configuração para Avaliação de Desempenho do Modelo

A biblioteca sklearn.metrics fornece um conjunto de métodos para avaliação do desempenho de modelos e datasets de aprendizagem de máquina.

Uma outra biblioteca também pode ser usada: https://huggingface.co/docs/evaluate/a_quick_tour

Função que retorna as métricas, a partir das predições e rótulos passados como parâmetro:

In [10]:
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    #compute precision, recall, and F1 score
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    #compute accuracy score
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, 
            "f1": f1,
            "precision": precision,
            "recall": recall
            }

## Treinamento do Modelo

Usaremos o modelo BERTimbau pré-treinado e os dados de treinamento do dataset para fazer um ajuste fino (finetune) do modelo para a tarefa de classificação de comentários como ofensivo ou não ofensivo.

Criação de um mapeamento dos ids e seus rótulos:

In [11]:
### INICIE O CÓDIGO AQUI ### (2 linhas de código)
id2label = {0: "não ofensivo", 1: "ofensivo"}
label2id = {label: id for id, label in id2label.items()}
### TERMINE O CÓDIGO AQUI ###

Configuração de uma função de otimização para definir um escalonamento de taxa de aprendizagem (learning rate) e alguns hiperparâmetros de treinamento.

Mais detalhes de como treinar um modelo TensorFlow com Keras: https://huggingface.co/docs/transformers/training#train-a-tensorflow-model-with-keras

In [12]:
batch_size = 16
#num_epochs = 3
num_epochs = 1
batches_per_epoch = len(tokenized_db["train"]) // batch_size
total_train_steps = int(batches_per_epoch * num_epochs)
optimizer, schedule = create_optimizer(init_lr=2e-5, num_warmup_steps=0, num_train_steps=total_train_steps)

2026-09-21 11:14:26.661906: I tensorflow/compiler/xla/stream_executor/cuda/cuda_gpu_executor.cc:894] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero. See more at https://github.com/torvalds/linux/blob/v6.0/Documentation/ABI/testing/sysfs-bus-pci#L344-L355
2026-09-21 11:14:26.664177: W tensorflow/core/common_runtime/gpu/gpu_device.cc:2211] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


Carrega o BERTimbau com TFAutoModelForSequenceClassification (https://huggingface.co/docs/transformers/main/en/model_doc/auto#transformers.TFAutoModelForSequenceClassification)

TFAutoModelForSequenceClassification é uma classe genérica de modelos que será instanciada com uma das classes de modelos da biblioteca (com um head de classificação de sequência) quando criado com o método from_pretrained() ou com o método from_config().

O head para classificação de sequência é uma camada linear no topo das saídas agrupadas (pooled) dos estados ocultos (hidden states) do modelo base (BERTimbau, nesse caso).

<img src="../notebooks-aula/figs/transformer-and-head.svg">
https://huggingface.co/course/chapter2/2?fw=tf

In [13]:
### INICIE O CÓDIGO AQUI ### (1 linha de código)
### Carrega o BERTimbau com TFAutoModelForSequenceClassification
model = TFAutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2, id2label=id2label, label2id=label2id)
### TERMINE O CÓDIGO AQUI ###

/home/marcos/projects/neural-networks-course/2 - Redes Neurais para Processamento de Linguagem Natural/.venv/lib/python3.10/site-packages/transformers/utils/generic.py:311: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  torch.utils._pytree._register_pytree_node(
All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier', 'bert/pooler/dense/bias:0', 'bert/pooler/dense/kernel:0']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Converte os datasets para o formato tf.data.Dataset com o prepare_tf_dataset().

In [14]:
tf_train_set = model.prepare_tf_dataset(
    tokenized_db["train"],
    shuffle=True,
    batch_size=batch_size,
    collate_fn=data_collator,
)

tf_validation_set = model.prepare_tf_dataset(
    tokenized_db["validation"],
    shuffle=False,
    batch_size=batch_size,
    collate_fn=data_collator,
)

tf_test_set = model.prepare_tf_dataset(
    tokenized_db["test"],
    shuffle=False,
    batch_size=batch_size,
    collate_fn=data_collator,
)

You're using a BertTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


Exibe a arquitetura e parâmetros do modelo:

In [15]:
model.summary()

Model: "tf_bert_for_sequence_classification"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  108923136 
                                                                 
 dropout_37 (Dropout)        multiple                  0         
                                                                 
 classifier (Dense)          multiple                  1538      
                                                                 
Total params: 108924674 (415.51 MB)
Trainable params: 108924674 (415.51 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


**SAÍDA ESPERADA:** <br>

Configura o modelo para treinamento:

In [16]:
model.compile(optimizer=optimizer)

Computa a acurácia das predições:

In [17]:
metric_callback = KerasMetricCallback(metric_fn=compute_metrics, eval_dataset=tf_validation_set)
callbacks = [metric_callback]

Chama o método "fit" para treinar o modelo com os datasets de treino e validação:

In [18]:
start_time = datetime.now()
# Treino
### INICIE O CÓDIGO AQUI ### (1 linha de código)
history = model.fit(tf_train_set, validation_data=tf_validation_set, epochs=num_epochs, callbacks=callbacks)
### TERMINE O CÓDIGO AQUI ###
# Tempo de treino
end_time = datetime.now()
training_time = (end_time - start_time).total_seconds()

280/280 [==============================] - 532s 2s/step - loss: 0.3331 - val_loss: 0.1952 - accuracy: 0.9223 - f1: 0.9223 - precision: 0.9233 - recall: 0.9223


In [19]:
print('Treinamento:')
print('Acurária : {:.1%}'.format(history.history['accuracy'][-1]))
#print('Acurária nos dados de validação: {:.1%}'.format(history.history['val_accuracy'][-1]))
print('Precisão : {:.1%}'.format(history.history['precision'][-1]))
print('Revocação: {:.1%}'.format(history.history['recall'][-1]))
print('F1       : {:.1%}'.format(history.history['f1'][-1]))
print('Tempo de treinamento: {:.1f}s (or {:.1f} minutes)'.format(training_time, training_time/60))

Treinamento:
Acurária : 92.2%
Precisão : 92.3%
Revocação: 92.2%
F1       : 92.2%
Tempo de treinamento: 531.8s (or 8.9 minutes)


Salva o modelo treinado:

In [20]:
dir_save_model = "./"+checkpoint+"-fine-tuned-hatebr-db"
tokenizer.save_pretrained(dir_save_model)
model.save_pretrained(dir_save_model)

## Avaliação do Modelo

Avalia o desempenho do modelo no conjunto de teste.

Na classificação multiclasse, *model.predict* retorna um vetor de probabilidades para cada classe. A função *argmax* retorna o índice do vetor com a maior probabilidade, indicando assim a classe predita.

In [21]:
def compute_metrics2(predictions, labels):
    #compute precision, recall, and F1 score
    precision, recall, f1, _ = precision_recall_fscore_support(labels, predictions, average='macro')
    #compute accuracy score
    acc = accuracy_score(labels, predictions)
    return {"accuracy": acc, 
            "f1": f1,
            "precision": precision,
            "recall": recall
            }

In [22]:
### INICIE O CÓDIGO AQUI ### (3 linhas de código)
output = model.predict(tf_test_set)
pred_labels = np.argmax(output.logits, axis=-1)
print(compute_metrics2(pred_labels, db["test"]["label"]))
### TERMINE O CÓDIGO AQUI ###

88/88 [==============================] - 33s 364ms/step
{'accuracy': 0.9157142857142857, 'f1': 0.9156585170602602, 'precision': 0.9168167234154008, 'recall': 0.9157142857142857}


In [23]:
# Imprime as primeiras instâncias
print("Predição: ", pred_labels[:20])
print("Correto:  ", db["test"]["label"][:20])
print("Texto:    ", db["test"]["instagram_comments"][:20])

Predição:  [1 1 1 1 1 1 1 1 1 1 1 0 1 1 1 1 1 1 1 1]
Correto:   [True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True, True]
Texto:     ['Mais um lixo', 'Essa mulher é doente.pilantra!', 'Vagabunda. Comunista. Mentirosa. O povo chileno nao merece uma desgraça desta.', 'Besta quadrada.', 'Hipocritas', 'Quem tem pena é galinha, mas ela é uma VACA LOUCA.', 'Mande essa Bachelet plantar batata no asfalto. Puta que pariu até o tamborete de bordel se acha no direito de falar do Brasil.', 'Cara de pau.', 'Porque é uma bandida', 'Tudo igual. Só pensam no próprio rabo', 'Cretina!', 'Quem não têm caráter é contra o presidente Bolsonaro', 'Será que alguém em sã consciência ainda vai acreditar nesta farsante? Não acredito em um fio do seu cabelo pintado.', 'Ela tem cara de homem né? Macron deve achar gata', 'Nojo dessa pilantra', 'Bando de canalhas', 'Vieja idiota.', 'Cretina!!!', 'Ridícula nojenta', 'Isso é uma realidade da imbecilidade h

## Predição

Prediz a saída de novos dados

<img src="../notebooks-aula/figs/transformer-pipeline.svg">
https://huggingface.co/course/chapter2/2?fw=tf

Vamos agora usar o modelo ajustado (finetuned) para fazer inferências.<br>
Cria um texto para que o modelo infira sua opinião:

In [24]:
### INICIE O CÓDIGO AQUI ### (1 linha de código)
text = "Esse governo é uma vergonha, só tem gente incompetente lá."
### TERMINE O CÓDIGO AQUI ###

Tokeniza o texto e retorna os tensores TensorFlow:

In [25]:
### INICIE O CÓDIGO AQUI ### (2 linhas de código)
inputs = tokenizer(text, return_tensors="tf")
inputs
### TERMINE O CÓDIGO AQUI ###

{'input_ids': <tf.Tensor: shape=(1, 17), dtype=int32, numpy=
array([[  101,  3758,  1161,   253,   230,   792, 12268,   117,  1203,
          376,  9349, 14644,   735,   403,  2920,   119,   102]],
      dtype=int32)>, 'token_type_ids': <tf.Tensor: shape=(1, 17), dtype=int32, numpy=array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]], dtype=int32)>, 'attention_mask': <tf.Tensor: shape=(1, 17), dtype=int32, numpy=array([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], dtype=int32)>}

<b>Logits</b> é o vetor de predição pura (não normalizado) que o modelo de classificação gera, o qual é comumente passado para uma função de normalização. Se o modelo está resolvendo um problema de classificação multiclasse, os logits tipicamente tornam-se uma entrada para a função softmax. E esta então gera um vetor de probabilidades (normalizado) com um valor para cada classe possível.

Passa as entradas para o modelo e retorna os logits:

In [26]:
model = TFAutoModelForSequenceClassification.from_pretrained(dir_save_model)
output_logits = model(**inputs).logits

Some layers from the model checkpoint at ./neuralmind/bert-base-portuguese-cased-fine-tuned-hatebr-db were not used when initializing TFBertForSequenceClassification: ['dropout_37']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at ./neuralmind/bert-base-portuguese-cased-fine-tuned-hatebr-db.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassification for predictions without further

Gera a classe predita a partir dos logits:

In [27]:
predicted_class_id = np.argmax(output_logits, axis=-1)[0]
model.config.id2label[predicted_class_id]

'ofensivo'

## Desafio

Modifique o classificador para predizer o nível de ofensividade (0 - não ofensivo, 1 - levemente ofensivo, 2 - moderadamente ofensivo, 3- altamente ofensivo). <br>
Isso significa que o atributo 'label' passará a ser o atributo 'offensiveness_levels'. <br>
Você vai precisar replicar todas as células do notebook, desde uma nova leitura do dataset, fazendo as devidas adaptações para a nova classe a ser predita. Agora não é mais uma classificação binária, pois o atributo classe tem 4 rótulos distintos.

Adicione abaixo a nova sequência de código.

In [28]:
### INICIE O CÓDIGO AQUI ### (várias linhas de código / várias células)

##### Pré-Processamento dos Dados

In [51]:
# Recarrega o dataset
db2 = load_dataset("ruanchaves/hatebr")

# Renomeia o atributo alvo (agora 'offensiveness_levels') para 'label'
db2 = db2.rename_column('offensiveness_levels', 'label')

# Printar os dados possíveis de 'offensiveness_levels'
valores = sorted(set(db2["train"]["label"]))
print(valores)

# Remove os atributos que não serão usados
db2 = db2.remove_columns([
    'offensive_language', 'antisemitism', 'apology_for_the_dictatorship',
    'fatphobia', 'homophobia', 'partyism', 'racism', 'religious_intolerance',
    'sexism', 'xenophobia', 'offensive_&_non-hate_speech', 'non-offensive',
    'specialist_1_hate_speech', 'specialist_2_hate_speech', 'specialist_3_hate_speech'
])

# Visualizar os dados de db2
db2["train"].to_pandas().head(10)

[0, 1, 2, 3]


,instagram_comments,label
0,este lixo ...,1
1,Comunista safada...,2
2,Comunista lixo!,1
3,Esquerdista.,2
4,Oportunista essa corrupta.Agora todos os Comun...,3
5,Plano da esquerda foi é e será sempre críticar...,2
6,O problema dela é que a tetinha secou,2
7,É uma cara de pau mesmo... 😡,3
8,Joga uma carteira de trabalho que correm todos...,1
9,BACHELETA DA TRETA E MUTRETA CHILENA..CANALHA ...,3


##### Tokenização pelo Modelo BERTimbau

In [30]:
# Reaproveitando o mesmo tokenizador e a mesma preprocess_function que já foi definida antes
tokenized_db2 = db2.map(preprocess_function, batched=True)
data_collator2 = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="tf")

Map: 100%|██████████| 4480/4480 [00:00<00:00, 77384.09 examples/s]


##### Treinamento do Modelo

Mapeamento dos 4 níveis de ofensividade

In [31]:
id2label2 = {0: "não ofensivo", 1: "levemente ofensivo", 2: "moderadamente ofensivo", 3: "altamente ofensivo"}
label2id2 = {label: id for id, label in id2label2.items()}

Configuração do otimizador

In [ ]:
batch_size2 = 16
num_epochs2 = 
batches_per_epoch2 = len(tokenized_db2["train"]) // batch_size2
total_train_steps2 = int(batches_per_epoch2 * num_epochs2)
optimizer2, schedule2 = create_optimizer(init_lr=2e-5, num_warmup_steps=0, num_train_steps=total_train_steps2)

Carregando o segundo modelo BERTimbau com os 4 rótulos de saída que vamos utilizar agora

In [33]:
model2 = TFAutoModelForSequenceClassification.from_pretrained(
    checkpoint, num_labels=4, id2label=id2label2, label2id=label2id2
)

model2.summary()

All model checkpoint layers were used when initializing TFBertForSequenceClassification.

Some layers of TFBertForSequenceClassification were not initialized from the model checkpoint at neuralmind/bert-base-portuguese-cased and are newly initialized: ['classifier', 'bert/pooler/dense/bias:0', 'bert/pooler/dense/kernel:0']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model: "tf_bert_for_sequence_classification_2"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 bert (TFBertMainLayer)      multiple                  108923136 
                                                                 
 dropout_113 (Dropout)       multiple                  0         
                                                                 
 classifier (Dense)          multiple                  3076      
                                                                 
Total params: 108926212 (415.52 MB)
Trainable params: 108926212 (415.52 MB)
Non-trainable params: 0 (0.00 Byte)
_________________________________________________________________


Convertendo os datasets para o formato tf.data.Dataset

In [34]:
tf_train_set2 = model2.prepare_tf_dataset(
    tokenized_db2["train"], shuffle=True, batch_size=batch_size2, collate_fn=data_collator2,
)
tf_validation_set2 = model2.prepare_tf_dataset(
    tokenized_db2["validation"], shuffle=False, batch_size=batch_size2, collate_fn=data_collator2,
)
tf_test_set2 = model2.prepare_tf_dataset(
    tokenized_db2["test"], shuffle=False, batch_size=batch_size2, collate_fn=data_collator2,
)

Compilando o segundo modelo com o otimizador criado

In [35]:
model2.compile(optimizer=optimizer2)

Treinando e avaliando primariamente o modelo

In [36]:
metric_callback2 = KerasMetricCallback(metric_fn=compute_metrics, eval_dataset=tf_validation_set2)
callbacks2 = [metric_callback2]

dist_train = Counter(db2["train"]["label"])
n_total = sum(dist_train.values())
n_classes = len(dist_train)

# peso inversamente proporcional à frequência: classes raras pesam mais
class_weight2 = {
    label: n_total / (n_classes * count)
    for label, count in dist_train.items()
}
print(class_weight2)

start_time2 = datetime.now()
history2 = model2.fit(
    tf_train_set2, 
    validation_data=tf_validation_set2, 
    epochs=num_epochs2, 
    callbacks=callbacks2,
    class_weight=class_weight2
)
end_time2 = datetime.now()
training_time2 = (end_time2 - start_time2).total_seconds()

print('Treinamento:')
print('Acurácia : {:.1%}'.format(history2.history['accuracy'][-1]))
print('Precisão : {:.1%}'.format(history2.history['precision'][-1]))
print('Revocação: {:.1%}'.format(history2.history['recall'][-1]))
print('F1       : {:.1%}'.format(history2.history['f1'][-1]))
print('Tempo de treinamento: {:.1f}s (ou {:.1f} minutos)'.format(training_time2, training_time2/60))

{1: 1.247216035634744, 2: 1.3160987074030552, 3: 2.281059063136456, 0: 0.5}
Epoch 1/3
280/280 [==============================] - 527s 2s/step - loss: 1.1276 - val_loss: 0.7528 - accuracy: 0.6589 - f1: 0.4667 - precision: 0.4969 - recall: 0.4793
Epoch 2/3
280/280 [==============================] - 516s 2s/step - loss: 0.9356 - val_loss: 0.7933 - accuracy: 0.6205 - f1: 0.4783 - precision: 0.4874 - recall: 0.4953
Epoch 3/3
280/280 [==============================] - 528s 2s/step - loss: 0.8241 - val_loss: 0.7834 - accuracy: 0.6357 - f1: 0.4793 - precision: 0.4919 - recall: 0.4950
Treinamento:
Acurácia : 63.6%
Precisão : 49.2%
Revocação: 49.5%
F1       : 47.9%
Tempo de treinamento: 1570.9s (ou 26.2 minutos)


Salvando modelo treinado

In [37]:
dir_save_model2 = "./"+checkpoint+"-fine-tuned-hatebr-offensiveness-levels"
tokenizer.save_pretrained(dir_save_model2)
model2.save_pretrained(dir_save_model2)

##### Avaliação do Modelo

In [38]:
output2 = model2.predict(tf_test_set2)
pred_labels2 = np.argmax(output2.logits, axis=-1)
print(compute_metrics2(pred_labels2, db2["test"]["label"]))

# Imprime as primeiras instâncias
print("Predição: ", pred_labels2[:20])
print("Correto:  ", db2["test"]["label"][:20])
print("Texto:    ", db2["test"]["instagram_comments"][:20])

88/88 [==============================] - 34s 366ms/step
{'accuracy': 0.6228571428571429, 'f1': 0.4617458610613227, 'precision': 0.469014194578503, 'recall': 0.4770929675724801}
Predição:  [3 3 3 1 2 3 3 2 3 2 1 0 1 1 3 3 3 1 3 1]
Correto:   [1, 3, 3, 2, 2, 3, 3, 3, 2, 2, 2, 2, 2, 2, 3, 3, 2, 3, 3, 2]
Texto:     ['Mais um lixo', 'Essa mulher é doente.pilantra!', 'Vagabunda. Comunista. Mentirosa. O povo chileno nao merece uma desgraça desta.', 'Besta quadrada.', 'Hipocritas', 'Quem tem pena é galinha, mas ela é uma VACA LOUCA.', 'Mande essa Bachelet plantar batata no asfalto. Puta que pariu até o tamborete de bordel se acha no direito de falar do Brasil.', 'Cara de pau.', 'Porque é uma bandida', 'Tudo igual. Só pensam no próprio rabo', 'Cretina!', 'Quem não têm caráter é contra o presidente Bolsonaro', 'Será que alguém em sã consciência ainda vai acreditar nesta farsante? Não acredito em um fio do seu cabelo pintado.', 'Ela tem cara de homem né? Macron deve achar gata', 'Nojo dessa pilan

##### Predição

In [39]:
textos_teste = {
    0: "Concordo com parte do que foi dito, mas acho que falta debater mais o tema.",        # não ofensivo
    1: "Esse projeto de lei é bem fraco, sinceramente.",                                     # levemente ofensivo
    2: "Só um bando de incompetente pra aprovar uma proposta dessas.",                       # moderadamente ofensivo
    3: "Mas que vergonha, esses políticos são a escória mais nojenta que existe.",           # altamente ofensivo
}

model2 = TFAutoModelForSequenceClassification.from_pretrained(dir_save_model2)

for nivel_esperado, texto in textos_teste.items():
    inputs2 = tokenizer(texto, return_tensors="tf")
    output_logits2 = model2(**inputs2).logits
    predicted_class_id2 = np.argmax(output_logits2, axis=-1)[0]
    predito = model2.config.id2label[predicted_class_id2]
    print(f"Esperado: {nivel_esperado} | Predito: {predito} | Texto: {texto}")

Some layers from the model checkpoint at ./neuralmind/bert-base-portuguese-cased-fine-tuned-hatebr-offensiveness-levels were not used when initializing TFBertForSequenceClassification: ['dropout_113']
- This IS expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing TFBertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
All the layers of TFBertForSequenceClassification were initialized from the model checkpoint at ./neuralmind/bert-base-portuguese-cased-fine-tuned-hatebr-offensiveness-levels.
If your task is similar to the task the model of the checkpoint was trained on, you can already use TFBertForSequenceClassific

Esperado: 0 | Predito: não ofensivo | Texto: Concordo com parte do que foi dito, mas acho que falta debater mais o tema.
Esperado: 1 | Predito: não ofensivo | Texto: Esse projeto de lei é bem fraco, sinceramente.
Esperado: 2 | Predito: altamente ofensivo | Texto: Só um bando de incompetente pra aprovar uma proposta dessas.
Esperado: 3 | Predito: altamente ofensivo | Texto: Mas que vergonha, esses políticos são a escória mais nojenta que existe.


In [40]:
### TERMINE O CÓDIGO AQUI ###

# Fim

Parabéns! Você efetuou todos os passos para criar um modelo de PLN para a tarefa de classificação de textos usando o Modelo BERT

-------------------------------------------